<h1><center>Adult income based classification benchmark<h1>

In [59]:
# The classification dataset with 2 classes, the target value is annual income above 50K or below 50K.
# Source of the dataset: https://archive.ics.uci.edu/dataset/2/adult

In [60]:
# The final weight estimate, who match this exact person's age, race, and sex.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedKFold, cross_validate

<h3><center>Loading and exploring the dataset<h3>

In [62]:
# I need to define name for columns, because the dataset has no header.
column_names = ["age", "workclass", "final_weight", "education", "education_num",
                "marital_status", "occupation", "relationship", "race", "sex",
                "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"]

# What I noticed, the dataset has ? mark so  I am gonna convert the ? mark into nan. 
df = pd.read_csv("Adult_inccome/adult.data", header=None,
                names=column_names, skipinitialspace=True, na_values="?")

print(f"Dataset shape: {df.shape}")

Dataset shape: (32561, 15)


In [63]:
df.head()

,age,workclass,final_weight,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [64]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       30725 non-null  str  
 2   final_weight    32561 non-null  int64
 3   education       32561 non-null  str  
 4   education_num   32561 non-null  int64
 5   marital_status  32561 non-null  str  
 6   occupation      30718 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital_gain    32561 non-null  int64
 11  capital_loss    32561 non-null  int64
 12  hours_per_week  32561 non-null  int64
 13  native_country  31978 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [65]:
# The DS has 6 numeric columns and 8 categorical columns. 
# The numeric_columns are: age, final_weight, education_num, capital_gain, capital_loss, hours_per_week
# The categorical_columns are: workclass, education, marital_status, occupation, relationship, race, sex, native_country

In [66]:
# Missing values
missing_values = df.isna().sum()
print(f"\nMissing values", missing_values[missing_values > 0])


Missing values workclass         1836
occupation        1843
native_country     583
dtype: int64


In [67]:
# let's check the annual income classes. 
In_come_info  = pd.DataFrame({'count': df['income'].value_counts(),'percent': df['income']
                              .value_counts(normalize=True).mul(100).round(2)})
In_come_info

,count,percent
income,,
<=50K,24720,75.92
>50K,7841,24.08


In [68]:
# Which columns are missing together?

missing_columns = ['workclass','occupation','native_country']
print('Total rows with missing value:',df[missing_columns].isna().any(axis=1).sum())
missing_pattern = (df[missing_columns].isna().value_counts().rename("rows"))

missing_pattern
# There are 4262 missing values and 2399 rows have them. 
# Every row missing workclass is also missing occupation: 1809 + 27 = 1836 and only 7 rows have missing occupation.  

Total rows with missing value: 2399


workclass  occupation  native_country
False      False       False             30162
True       True        False              1809
False      False       True                556
True       True        True                 27
False      True        False                 7
Name: rows, dtype: int64

In [69]:
# Compring missing values with target values: 
missing_data = (df[missing_columns].isna().any(axis=1))
print(pd.crosstab(missing_data,df["income"],margins=True))
# Most od missing values are associated with under 50K
# If I run df.dropna(), it would remove 2399 rows, around 7.3% of the data and change the class distribution overall. 
# mmm, I am gonna keep them with missing values, later I will use constant value  inside the categorica preprocessing pipeline. 

income  <=50K  >50K    All
row_0                     
False   22654  7508  30162
True     2066   333   2399
All     24720  7841  32561


In [70]:
# Let's check duplicated 
duplicate_rows = df[df.duplicated(keep=False)]
print("Rows in duplicate groups:",len(duplicate_rows))
print(duplicate_rows["income"].value_counts())

Rows in duplicate groups: 47
income
<=50K    43
>50K      4
Name: count, dtype: int64


In [71]:
# Let's check education redundancy
# According to dataset description, education number is number code for each education level. 
education_mapping = (df[["education", "education_num"]].drop_duplicates().sort_values("education_num"))
education_mapping

,education,education_num
224,Preschool,1
160,1st-4th,2
56,5th-6th,3
15,7th-8th,4
6,9th,5
77,10th,6
3,11th,7
415,12th,8
2,HS-grad,9
10,Some-college,10


In [72]:
# Here I am gonna check two things:
# 1- for each text label, how many different number/code for education level does it have? 
# 2- how many text label map to more than one number? 
education_to_num = df.groupby('education')['education_num'].nunique()
num_to_education = df.groupby('education_num')['education'].nunique()

print('Education categories with multiple numbers:',(education_to_num > 1).sum())
print("education numbers with multiple categories:",(num_to_education>1).sum())
# That is a good singn, every education level has excactly one code number and every education number has exactly one education lovel.
# I am gonna keep educatiom level and remove education number, because the model treats the  education number as regular number line.
# For example, form 1 to 2 = preschool to 1st-4th and from 14 to 15 masters to prof-school.
# But in real,  life those gaps are not equal.
df = df.drop('education_num', axis=1)

Education categories with multiple numbers: 0
education numbers with multiple categories: 0


In [73]:
# Let's check the numeric ranges,
numeric_columns = ["age", "capital_gain", "capital_loss", "hours_per_week"]
df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
age,32561.0,38.581647,13.640433,17.0,28.0,37.0,48.0,90.0
capital_gain,32561.0,1077.648844,7385.292085,0.0,0.0,0.0,0.0,99999.0
capital_loss,32561.0,87.303830,402.960219,0.0,0.0,0.0,0.0,4356.0
hours_per_week,32561.0,40.437456,12.347429,1.0,40.0,40.0,45.0,99.0


In [74]:
# I am gonna drop the final_weight, becausue it is not ordinary personal characteristic such as age or working hours.
df = df.drop("final_weight", axis=1)

In [75]:
zero_percent = (df[["capital_gain", "capital_loss"]].eq(0).mean().mul(100).round(2))
print('Zero values percentage:')
zero_percent
# Most of values are zero 
# It seems, the distribution is right skewed. 

Zero values percentage:


capital_gain    91.67
capital_loss    95.33
dtype: float64

In [76]:
# Loading and exploring the test dataset. 

test_df = pd.read_csv("Adult_inccome/adult.test",header=None,
                      names=column_names, skipinitialspace=True, na_values="?",skiprows=1)

test_df['income'] = test_df["income"].str.replace(".", "",regex=False)

print("Test shape:", test_df.shape)
print("\nTest target values:")
print(test_df["income"].value_counts())

print("\nTest missing values:")
print(test_df.isna().sum()[test_df.isna().sum() > 0])

Test shape: (16281, 15)

Test target values:
income
<=50K    12435
>50K      3846
Name: count, dtype: int64

Test missing values:
workclass         963
occupation        966
native_country    274
dtype: int64


In [77]:
test_df = test_df.drop(['final_weight', "education_num"],axis=1)

print("Training shape:", df.shape)
print("Test shape:", test_df.shape)

Training shape: (32561, 13)
Test shape: (16281, 13)


<h3><center>Building preprocessing pipeline<h3>

In [78]:
# Let's define  predictors and target
X_train = df.drop('income',axis=1)
y_train = df["income"]

X_test = test_df.drop("income", axis=1)
y_test = test_df["income"]

print('X_train shape:', X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (32561, 12)
X_test shape: (16281, 12)


In [ ]:
# I defined already the numeric columns in previous cell. 

categorical_columns = ["workclass", "education", "marital_status","occupation",
                       "relationship", "race", "sex", "native_country"]

In [80]:
# In here, I am gonna build categorical pipeline.

categorical_preprocessor = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
                                    ("encoder", OneHotEncoder(handle_unknown="ignore"))])
print(categorical_preprocessor)

Pipeline(steps=[('imputer',
                 SimpleImputer(fill_value='Unknown', strategy='constant')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))])


In [81]:
# In here, I am gonna build pipeline for numeric columns.
linear_preprocessor = ColumnTransformer([("numeric", StandardScaler(), numeric_columns),
                                         ("categorical", categorical_preprocessor, categorical_columns)])

print(linear_preprocessor)

ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 ['age', 'capital_gain', 'capital_loss',
                                  'hours_per_week']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Unknown',
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['workclass', 'education', 'marital_status',
                                  'occupation', 'relationship', 'race', 'sex',
                                  'native_country'])])


In [ ]:
X_train_ready = linear_preprocessor.fit_transform(X_train)
X_test_ready = linear_preprocessor.transform(X_test)

print("transformed training shape:", X_train.shape)
print('Transformed test shape:', X_test.shape)

transformed training shape: (32561, 12)
Transformed test shape: (16281, 12)


In [84]:
# Now, let's covert the target column to binary values
target_mapping = {'<=50K': 0, '>50K': 1}

y_train_binary = y_train.map(target_mapping)
y_test_binary = y_test.map(target_mapping)

print("Training target:")
print(y_train.value_counts())

print("\nMissing mapped labels:")
print(y_train.isna().sum())

Training target:
income
<=50K    24720
>50K      7841
Name: count, dtype: int64

Missing mapped labels:
0


In [ ]:
# In this cell, I am gonna check, does the real model perform better than simply predicting the majority class?
dummy_pipeline = Pipeline([("preprocessor", linear_preprocessor),
                           ("model", DummyClassifier(strategy="most_frequent"))])


logistic_pipeline = Pipeline([("preprocessor", linear_preprocessor),
                              ("model", LogisticRegression(max_iter=1000))])
print(dummy_pipeline)
print(logistic_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['age', 'capital_gain',
                                                   'capital_loss',
                                                   'hours_per_week']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Unknown',
                                                                                 strategy='constant')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['workclass', 'education',
                                                   'marital_s

In [ ]:
# defining  cross validation and metrics
cv = StratifiedKFold(n_splits=5,shuffle=True, random_state=42)

scoring = ["accuracy", "balanced_accuracy", "precision",
           "recall", "f1", "roc_auc", "average_precision"]

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision']


In [91]:
# Now, let's run the benchmark  
models = {"Dummy": dummy_pipeline,
          "Logistic Regression": logistic_pipeline}

cv_results = {}

for model_name, model in models.items():
    scores = cross_validate(model, X_train, y_train_binary, cv=cv, scoring=scoring, n_jobs=-1)

    cv_results[model_name] = {
        metric: scores[f"test_{metric}"].mean()
        for metric in scoring}

benchmark_results = pd.DataFrame(cv_results).T

print(benchmark_results.round(4))

                     accuracy  balanced_accuracy  precision  recall     f1  \
Dummy                  0.7592              0.500     0.0000  0.0000  0.000   
Logistic Regression    0.8516              0.766     0.7345  0.6011  0.661   

                     roc_auc  average_precision  
Dummy                 0.5000             0.2408  
Logistic Regression   0.9066             0.7666  
